# cAPTure: four-model development comparison

This notebook compares the completed seed-42 Edge MLP, EdgeGRU, StaticGNN, and ST-GNN jobs across both development folds. It verifies immutable artifacts and exact OOF alignment, then reports ranking, operational, early-warning, scenario, step, and resource results.

This is an exploratory partial comparison. It performs no training and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`. A topology-centered claim remains unavailable until the matched ST-GNN control without GAT message passing is complete.

## 1. Prepare the Colab environment

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import Markdown, display
from utils.capture_graph_comparison import (
    run_graph_four_model_comparison,
    validate_graph_four_model_comparison,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("A GPU runtime is not required for this evaluation.")

Mounted at /content/drive
Repository commit: 16e39c812f80dfb7aa04aeb3359bc2cbb16c28ef
A GPU runtime is not required for this evaluation.


## 2. Bind the completed training jobs

The comparison is bound to the authorized graph-training run. All eight jobs listed below must already be complete.

In [2]:
BINDING_RUN_ID = "20260929T144234_934044Z_graph_training_binding"
BINDING_DIR = DRIVE_ROOT / "graph_training_binding_runs" / BINDING_RUN_ID
TRAINING_ROOT = DRIVE_ROOT / "graph_training_runs" / BINDING_RUN_ID
TRAINING_CONFIG_PATH = PROJECT_ROOT / "configs/capture_graph_training_v1.yaml"
MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
TERMINAL_POLICY_PATH = PROJECT_ROOT / "configs/capture_early_warning_audit_v1.yaml"

MODEL_NAMES = ["edge_mlp", "edge_gru", "static_gnn", "st_gnn"]
FOLDS = ["A", "B"]
required_paths = [
    BINDING_DIR, TRAINING_ROOT, TRAINING_CONFIG_PATH, MANIFEST_PATH,
    TERMINAL_POLICY_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required comparison inputs are missing: {missing}")

inventory = []
for model_name in MODEL_NAMES:
    for fold in FOLDS:
        job_id = f"{model_name}__fold_{fold}"
        completion_path = TRAINING_ROOT / job_id / "completion.json"
        completion = (
            json.loads(completion_path.read_text(encoding="utf-8"))
            if completion_path.is_file() else {}
        )
        inventory.append({
            "job_id": job_id,
            "completion_exists": completion_path.is_file(),
            "status": completion.get("status"),
            "best_epoch_count": completion.get("best_epoch_count"),
            "held_out_scenarios_accessed": completion.get("held_out_scenarios_accessed"),
        })
inventory_table = pd.DataFrame(inventory)
display(inventory_table)
if not inventory_table["completion_exists"].all() or not (inventory_table["status"] == "complete").all():
    raise RuntimeError("All eight model/fold jobs must be complete before comparison.")

,job_id,completion_exists,status,best_epoch_count,held_out_scenarios_accessed
0,edge_mlp__fold_A,True,complete,1,False
1,edge_mlp__fold_B,True,complete,21,False
2,edge_gru__fold_A,True,complete,11,False
3,edge_gru__fold_B,True,complete,1,False
4,static_gnn__fold_A,True,complete,12,False
5,static_gnn__fold_B,True,complete,3,False
6,st_gnn__fold_A,True,complete,3,False
7,st_gnn__fold_B,True,complete,3,False


## 3. Create or verify the immutable comparison

The first run verifies every source checksum and compares packet keys, timestamps, labels, steps, and sequence IDs across models before computing metrics. Set `RUN_COMPARISON=True` only after reviewing the inventory. A normal rerun with the same run ID validates and reuses the completed report.

In [4]:
COMPARISON_RUN_ID = None  # Set this to reuse a completed comparison.
RUN_COMPARISON = True
if COMPARISON_RUN_ID is None:
    COMPARISON_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_four_model_comparison"
    )
OUTPUT_DIR = DRIVE_ROOT / "graph_comparison_runs" / COMPARISON_RUN_ID
print("Comparison run ID:", COMPARISON_RUN_ID)
print("Comparison output:", OUTPUT_DIR)

if OUTPUT_DIR.exists():
    comparison_report = validate_graph_four_model_comparison(
        output_dir=OUTPUT_DIR,
        training_root=TRAINING_ROOT,
        binding_dir=BINDING_DIR,
        training_config_path=TRAINING_CONFIG_PATH,
        manifest_path=MANIFEST_PATH,
        terminal_policy_path=TERMINAL_POLICY_PATH,
    )
elif RUN_COMPARISON:
    comparison_report = run_graph_four_model_comparison(
        training_root=TRAINING_ROOT,
        binding_dir=BINDING_DIR,
        training_config_path=TRAINING_CONFIG_PATH,
        manifest_path=MANIFEST_PATH,
        terminal_policy_path=TERMINAL_POLICY_PATH,
        output_dir=OUTPUT_DIR,
        batch_size=250_000,
    )
else:
    comparison_report = None
    print("Set RUN_COMPARISON=True to start the read-only OOF evaluation.")

Comparison run ID: 20261001T153351_608797Z_graph_four_model_comparison
Comparison output: /content/drive/MyDrive/capture_gate0/graph_comparison_runs/20261001T153351_608797Z_graph_four_model_comparison
Evaluating immutable OOF predictions for edge_mlp...
Evaluating immutable OOF predictions for edge_gru...
Evaluating immutable OOF predictions for static_gnn...
Evaluating immutable OOF predictions for st_gnn...


## 4. Ranking comparison

The primary aggregation first averages scenarios within each outer fold and then gives equal weight to the two folds. The flat mean across all five scenarios is retained only as a diagnostic.

In [5]:
if comparison_report is None:
    raise RuntimeError("Run or load the comparison before displaying results.")
ranking_summary = pd.DataFrame(comparison_report["tables"]["ranking_summary"])
ranking_by_scenario = pd.DataFrame(
    comparison_report["tables"]["ranking_by_scenario"]
)
display(ranking_summary.set_index("model_display")[[
    "average_precision", "roc_auc",
    "flat_average_precision_diagnostic", "flat_roc_auc_diagnostic",
]])
display(ranking_by_scenario.set_index(["model_display", "scenario"])[[
    "fold", "packets", "normal_packets", "attack_packets",
    "average_precision", "roc_auc",
]])

,average_precision,roc_auc,flat_average_precision_diagnostic,flat_roc_auc_diagnostic
model_display,,,,
Edge MLP,0.959441,0.958992,0.956629,0.956504
EdgeGRU,0.993317,0.996617,0.993013,0.996770
StaticGNN,0.994474,0.996179,0.994318,0.995982
ST-GNN,0.992172,0.997241,0.990720,0.996763


fold  packets  normal_packets  attack_packets  \
model_display scenario                                                          
Edge MLP      train_dollar_char    A  2882555         1705003         1177552   
              train_slash_char     A  6425516         1705003         4720513   
              train_sub_exf        A  2225807         1705003          520804   
              train_empty_conn     B  1175779          746806          428973   
              train_qos_mid        B  1499717          746806          752911   
EdgeGRU       train_dollar_char    A  2882555         1705003         1177552   
              train_slash_char     A  6425516         1705003         4720513   
              train_sub_exf        A  2225807         1705003          520804   
              train_empty_conn     B  1175779          746806          428973   
              train_qos_mid        B  1499717          746806          752911   
StaticGNN     train_dollar_char    A  2882555         1705003         1177552   
              train_slash_char     A  6425516         1705003         4720513   
              train_sub_exf        A  2225807         1705003          520804   
              train_empty_conn     B  1175779          746806          428973   
              train_qos_mid        B  1499717          746806          752911   
ST-GNN        train_dollar_char    A  2882555         1705003         1177552   
              train_slash_char     A  6425516         1705003         4720513   
              train_sub_exf        A  2225807         1705003          520804   
              train_empty_conn     B  1175779          746806          428973   
              train_qos_mid        B  1499717          746806          752911   

                                 average_precision   roc_auc  
model_display scenario                                        
Edge MLP      train_dollar_char           0.895628  0.879394  
              train_slash_char            0.994230  0.983640  
              train_sub_exf               0.946282  0.976629  
              train_empty_conn            0.982295  0.989144  
              train_qos_mid               0.964709  0.953715  
EdgeGRU       train_dollar_char           0.987616  0.995243  
              train_slash_char            0.999031  0.998657  
              train_sub_exf               0.988747  0.998236  
              train_empty_conn            0.995354  0.997113  
              train_qos_mid               0.994317  0.994600  
StaticGNN     train_dollar_char           0.981764  0.985893  
              train_slash_char            0.999985  0.999958  
              train_sub_exf               0.999331  0.999740  
              train_empty_conn            0.991376  0.995176  
              train_qos_mid               0.999135  0.999145  
ST-GNN        train_dollar_char           0.983321  0.992849  
              train_slash_char            0.993922  0.995400  
              train_sub_exf               0.977491  0.996304  
              train_empty_conn            0.999043  0.999441  
              train_qos_mid               0.999823  0.999821

## 5. Primary operational comparison

Each model receives its own development-OOF threshold under the frozen budget of one false-alert window per hour. A timely iteration requires the first correct alert window to close strictly before the final malicious packet. Preterminal coverage requires a correct alert strictly before the first declared terminal-action packet. Because threshold calibration and reporting reuse the same development OOF predictions, these are screening estimates rather than confirmatory estimates.

In [6]:
operational_summary = pd.DataFrame(
    comparison_report["tables"]["primary_operational_summary"]
)
operational_by_scenario = pd.DataFrame(
    comparison_report["tables"]["primary_operational_by_scenario"]
)
display(operational_summary.set_index("model_display")[[
    "threshold", "worst_fold_false_alert_windows_per_hour",
    "packet_precision", "packet_recall", "packet_f1", "packet_f2",
    "sequence_detection_rate",
    "timely_iteration_rate", "early_before_terminal_action",
    "median_seconds_before_iteration_end",
    "median_seconds_before_terminal_action",
]])
display(operational_by_scenario.set_index(["model_display", "scenario"])[[
    "fold", "false_alert_windows_per_hour", "packet_precision",
    "packet_recall", "packet_f1", "packet_f2",
    "iterations", "detected_iterations",
    "timely_iterations", "timely_iteration_rate",
    "early_before_terminal_action", "seconds_before_terminal_action",
]])

,threshold,worst_fold_false_alert_windows_per_hour,packet_precision,packet_recall,packet_f1,packet_f2,sequence_detection_rate,timely_iteration_rate,early_before_terminal_action,median_seconds_before_iteration_end,median_seconds_before_terminal_action
model_display,,,,,,,,,,,
Edge MLP,0.999966,0.000000,NaN,0.273171,0.353291,0.300421,0.170342,0.135268,0.5,117.232691,4453.556418
EdgeGRU,1.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,NaN
StaticGNN,0.999141,0.946445,0.999746,0.705226,0.808529,0.740817,0.740541,0.714158,1.0,53.193836,3781.753496
ST-GNN,1.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,NaN


fold  false_alert_windows_per_hour  \
model_display scenario                                               
Edge MLP      train_dollar_char    A                      0.000000   
              train_slash_char     A                      0.000000   
              train_sub_exf        A                      0.000000   
              train_empty_conn     B                      0.000000   
              train_qos_mid        B                      0.000000   
EdgeGRU       train_dollar_char    A                      0.000000   
              train_slash_char     A                      0.000000   
              train_sub_exf        A                      0.000000   
              train_empty_conn     B                      0.000000   
              train_qos_mid        B                      0.000000   
StaticGNN     train_dollar_char    A                      0.743162   
              train_slash_char     A                      0.868778   
              train_sub_exf        A                      1.186339   
              train_empty_conn     B                      1.238938   
              train_qos_mid        B                      0.653951   
ST-GNN        train_dollar_char    A                      0.000000   
              train_slash_char     A                      0.000000   
              train_sub_exf        A                      0.000000   
              train_empty_conn     B                      0.000000   
              train_qos_mid        B                      0.000000   

                                 packet_precision  packet_recall  packet_f1  \
model_display scenario                                                        
Edge MLP      train_dollar_char               NaN       0.000000   0.000000   
              train_slash_char                NaN       0.000000   0.000000   
              train_sub_exf                   NaN       0.000000   0.000000   
              train_empty_conn           1.000000       0.555375   0.714137   
              train_qos_mid              1.000000       0.537311   0.699027   
EdgeGRU       train_dollar_char               NaN       0.000000   0.000000   
              train_slash_char                NaN       0.000000   0.000000   
              train_sub_exf                   NaN       0.000000   0.000000   
              train_empty_conn                NaN       0.000000   0.000000   
              train_qos_mid                   NaN       0.000000   0.000000   
StaticGNN     train_dollar_char          0.998727       0.368509   0.538370   
              train_slash_char           0.999989       0.975710   0.987700   
              train_sub_exf              0.999891       0.965192   0.982235   
              train_empty_conn           0.999945       0.639383   0.780012   
              train_qos_mid              0.999969       0.641915   0.781901   
ST-GNN        train_dollar_char               NaN       0.000000   0.000000   
              train_slash_char                NaN       0.000000   0.000000   
              train_sub_exf                   NaN       0.000000   0.000000   
              train_empty_conn                NaN       0.000000   0.000000   
              train_qos_mid                   NaN       0.000000   0.000000   

                                 packet_f2  iterations  detected_iterations  \
model_display scenario                                                        
Edge MLP      train_dollar_char   0.000000         232                    0   
              train_slash_char    0.000000         356                    0   
              train_sub_exf       0.000000         335                    0   
              train_empty_conn    0.609582         107                   46   
              train_qos_mid       0.592102         171                   43   
EdgeGRU       train_dollar_char   0.000000         232                    0   
              train_slash_char    0.000000         356                    0   
              train_sub_exf       0.00000

## 6. False-alert-budget sensitivity

One false-alert window per hour remains the primary budget. The stricter one-per-12-hours budget and the more permissive one-per-five-minutes budget were frozen before this comparison and are reported as sensitivity analyses. They must not be used to replace the primary budget after inspecting results.

In [7]:
budget_sensitivity = pd.DataFrame(
    comparison_report["tables"]["budget_sensitivity_summary"]
)
budget_sensitivity_by_scenario = pd.DataFrame(
    comparison_report["tables"]["budget_sensitivity_by_scenario"]
)
display(budget_sensitivity.set_index(["model_display", "budget"])[[
    "target_false_alert_windows_per_hour",
    "worst_fold_false_alert_windows_per_hour", "threshold",
    "packet_precision", "packet_recall", "packet_f1", "packet_f2",
    "sequence_detection_rate", "timely_iteration_rate",
    "early_before_terminal_action",
]])
display(budget_sensitivity_by_scenario.set_index(
    ["model_display", "budget", "scenario"]
)[[
    "fold", "false_alert_windows_per_hour", "packet_precision",
    "packet_recall", "packet_f1", "packet_f2",
    "sequence_detection_rate", "timely_iteration_rate",
]])

target_false_alert_windows_per_hour  \
model_display budget                                                   
Edge MLP      one_per_hour                                  1.000000   
              one_per_12_hours                              0.083333   
              one_per_5_minutes                            12.000000   
EdgeGRU       one_per_hour                                  1.000000   
              one_per_12_hours                              0.083333   
              one_per_5_minutes                            12.000000   
StaticGNN     one_per_hour                                  1.000000   
              one_per_12_hours                              0.083333   
              one_per_5_minutes                            12.000000   
ST-GNN        one_per_hour                                  1.000000   
              one_per_12_hours                              0.083333   
              one_per_5_minutes                            12.000000   

                                 worst_fold_false_alert_windows_per_hour  \
model_display budget                                                       
Edge MLP      one_per_hour                                      0.000000   
              one_per_12_hours                                  0.000000   
              one_per_5_minutes                                 8.593168   
EdgeGRU       one_per_hour                                      0.000000   
              one_per_12_hours                                  0.000000   
              one_per_5_minutes                                 0.000000   
StaticGNN     one_per_hour                                      0.946445   
              one_per_12_hours                                  0.000000   
              one_per_5_minutes                                11.970775   
ST-GNN        one_per_hour                                      0.000000   
              one_per_12_hours                                  0.000000   
              one_per_5_minutes                                 0.000000   

                                 threshold  packet_precision  packet_recall  \
model_display budget                                                          
Edge MLP      one_per_hour        0.999966               NaN       0.273171   
              one_per_12_hours    0.999966               NaN       0.273171   
              one_per_5_minutes   0.937334          0.998358       0.710962   
EdgeGRU       one_per_hour        1.000000               NaN       0.000000   
              one_per_12_hours    1.000000               NaN       0.000000   
              one_per_5_minutes   1.000000               NaN       0.000000   
StaticGNN     one_per_hour        0.999141          0.999746       0.705226   
              one_per_12_hours    1.000000               NaN       0.186682   
              one_per_5_minutes   0.966401          0.998293       0.793026   
ST-GNN        one_per_hour        1.000000               NaN       0.000000   
              one_per_12_hours    1.000000               NaN       0.000000   
              one_per_5_minutes   1.000000               NaN       0.000000   

                                 packet_f1  packet_f2  \
model_display budget                                    
Edge MLP      one_per_hour        0.353291   0.300421   
              one_per_12_hours    0.353291   0.300421   
              one_per_5_minutes   0.816935   0.748571   
EdgeGRU       one_per_hour        0.000000   0.000000   
              one_per_12_hours    0.000000   0.000000   
              one_per_5_minutes   0.000000   0.000000   
StaticGNN     one_per_hour        0.808529   0.740817   
              one_per_12_hours    0.265714   0.211616   
              one_per_5_minutes   0.865048   0.818163   
ST-GNN        one_per_hour        0.000000   0.000000   
              one_per_12_hours    0.000000   0.000000   
              one_per_5_minutes   0.000000   0.000000   

                                 sequence_detection_r

fold  \
model_display budget            scenario                 
Edge MLP      one_per_hour      train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
              one_per_12_hours  train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
              one_per_5_minutes train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
EdgeGRU       one_per_hour      train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
              one_per_12_hours  train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
              one_per_5_minutes train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
StaticGNN     one_per_hour      train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
              one_per_12_hours  train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
              one_per_5_minutes train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
ST-GNN        one_per_hour      train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
              one_per_12_hours  train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   
              one_per_5_minutes train_dollar_char    A   
                                train_slash_char     A   
                                train_sub_exf        A   
                                train_empty_conn     B   
                                train_qos_mid        B   

                                                   false_alert_windows_per_hour  \
model_display budget            scenario                                          
Edge MLP      one_per_hour      train_dollar_char                      0.000000   
                                train_slash_char                       0.000000   
                                train_sub_exf                          0.000000   
                                train_

## 7. Predeclared pairwise contrasts

`StaticGNN - Edge MLP` is the current aggregation/message-passing signal without recurrent memory. `ST-GNN - EdgeGRU` is the temporal comparison, but it does not perfectly isolate GAT because the model families still differ. The pending matched ST-GNN control disables GATv2 layers while retaining endpoint aggregation and per-node memory.

In [8]:
pairwise = pd.DataFrame(comparison_report["tables"]["pairwise_comparisons"])
display(pairwise.set_index(["comparison", "metric"])[[
    "candidate", "control", "candidate_hierarchical_macro",
    "control_hierarchical_macro", "hierarchical_macro_delta",
    "frozen_practical_margin", "scenarios_with_positive_delta",
    "scenarios_with_zero_delta", "scenarios_with_negative_delta",
]])

candidate  \
comparison                             metric                                     
temporal_memory_without_gat            average_precision               edge_gru   
                                       roc_auc                         edge_gru   
                                       packet_recall                   edge_gru   
                                       packet_f1                       edge_gru   
                                       packet_f2                       edge_gru   
                                       sequence_detection_rate         edge_gru   
                                       timely_iteration_rate           edge_gru   
                                       early_before_terminal_action    edge_gru   
current_message_passing_without_memory average_precision             static_gnn   
                                       roc_auc                       static_gnn   
                                       packet_recall                 static_gnn   
                                       packet_f1                     static_gnn   
                                       packet_f2                     static_gnn   
                                       sequence_detection_rate       static_gnn   
                                       timely_iteration_rate         static_gnn   
                                       early_before_terminal_action  static_gnn   
temporal_memory_with_gat               average_precision                 st_gnn   
                                       roc_auc                           st_gnn   
                                       packet_recall                     st_gnn   
                                       packet_f1                         st_gnn   
                                       packet_f2                         st_gnn   
                                       sequence_detection_rate           st_gnn   
                                       timely_iteration_rate             st_gnn   
                                       early_before_terminal_action      st_gnn   
gat_signal_in_temporal_models          average_precision                 st_gnn   
                                       roc_auc                           st_gnn   
                                       packet_recall                     st_gnn   
                                       packet_f1                         st_gnn   
                                       packet_f2                         st_gnn   
                                       sequence_detection_rate           st_gnn   
                                       timely_iteration_rate             st_gnn   
                                       early_before_terminal_action      st_gnn   

                                                                        control  \
comparison                             metric                                     
temporal_memory_without_gat            average_precision               edge_mlp   
                                       roc_auc                         edge_mlp   
                                       packet_recall                   edge_mlp   
                                       packet_f1                       edge_mlp   
                                       packet_f2                       edge_mlp   
                                       sequence_detection_rate         edge_mlp   
                                       timely_iteration_rate           edge_mlp   
                                       early_before_terminal_action    edge_mlp   
current_message_passing_without_memory average_precision               edge_mlp   
                                       roc_auc                         edge_mlp   
                                       packet_recall                   edge_mlp   
                                       packet_f1                       edge_mlp   
                                       packet_f2                       edge_mlp   
  

## 8. Step breakdown and resource cost

In [9]:
step_metrics = pd.DataFrame(comparison_report["tables"]["primary_step_metrics"])
resource_summary = pd.DataFrame(comparison_report["tables"]["resource_summary"])
resource_by_job = pd.DataFrame(comparison_report["tables"]["resource_by_job"])
display(step_metrics.set_index(["model_display", "scenario", "attack_step"])[[
    "fold", "iterations", "detected_iterations", "timely_iterations",
    "sequence_detection_rate", "timely_iteration_rate",
    "median_timely_lead_seconds",
]])
display(resource_summary.set_index("model_display")[[
    "best_epoch_count_fold_A", "best_epoch_count_fold_B", "parameters",
    "total_wall_seconds", "seconds_per_million_packets",
    "maximum_peak_host_rss_gib", "maximum_peak_cuda_allocated_gib",
]])
display(resource_by_job.set_index(["model_display", "fold"]))

fold  iterations  \
model_display scenario          attack_step                              
Edge MLP      train_dollar_char brute_force_malformed    A           1   
                                dollar_char              A         166   
                                mqtt_cat                 A           7   
                                nmap_10_T5               A           9   
                                nmap_banner              A          14   
...                                                    ...         ...   
ST-GNN        train_qos_mid     nmap_mqtt                B           9   
                                nmap_sub                 B          12   
                                qos_mid                  B          39   
                                qos_mid_ddos             B          75   
                                scp_inst                 B           5   

                                                       detected_iterations  \
model_display scenario          attack_step                                  
Edge MLP      train_dollar_char brute_force_malformed                    0   
                                dollar_char                              0   
                                mqtt_cat                                 0   
                                nmap_10_T5                               0   
                                nmap_banner                              0   
...                                                                    ...   
ST-GNN        train_qos_mid     nmap_mqtt                                0   
                                nmap_sub                                 0   
                                qos_mid                                  0   
                                qos_mid_ddos                             0   
                                scp_inst                                 0   

                                                       timely_iterations  \
model_display scenario          attack_step                                
Edge MLP      train_dollar_char brute_force_malformed                  0   
                                dollar_char                            0   
                                mqtt_cat                               0   
                                nmap_10_T5                             0   
                                nmap_banner                            0   
...                                                                  ...   
ST-GNN        train_qos_mid     nmap_mqtt                              0   
                                nmap_sub                               0   
                                qos_mid                                0   
                                qos_mid_ddos                           0   
                                scp_inst                               0   

                                                       sequence_detection_rate  \
model_display scenario          attack_step                                      
Edge MLP      train_dollar_char brute_force_malformed                      0.0   
                                dollar_char                                0.0   
                                mqtt_cat                                   0.0   
                                nmap_10_T5                                 0.0   
                                nmap_banner                                0.0   
...                                                                        ...   
ST-GNN        train_qos_mid     nmap_mqtt                                  0.0   
                                nmap_sub                                   0.0   
                                qos_mid                                    0.0   
                                qos_mid_ddos                               0.0   
                                scp_inst                                   0.0   

                    

,best_epoch_count_fold_A,best_epoch_count_fold_B,parameters,total_wall_seconds,seconds_per_million_packets,maximum_peak_host_rss_gib,maximum_peak_cuda_allocated_gib
model_display,,,,,,,
Edge MLP,1,21,11137,4172.588271,5.277319,2.099953,0.118065
EdgeGRU,11,1,50946,5824.838758,10.810221,2.318100,0.190306
StaticGNN,12,3,55745,9467.906064,18.199225,2.313602,0.273281
ST-GNN,3,3,80706,9254.983488,21.482937,2.327396,0.275872


model  best_epoch_count  parameters  checkpoint_mib  \
model_display fold                                                             
Edge MLP      A       edge_mlp                 1       11137        0.046941   
              B       edge_mlp                21       11137        0.046941   
EdgeGRU       A       edge_gru                11       50946        0.200783   
              B       edge_gru                 1       50946        0.200783   
StaticGNN     A     static_gnn                12       55745        0.221395   
              B     static_gnn                 3       55745        0.221395   
ST-GNN        A         st_gnn                 3       80706        0.318159   
              B         st_gnn                 3       80706        0.318159   

                    wall_seconds  peak_host_rss_gib  peak_cuda_allocated_gib  \
model_display fold                                                             
Edge MLP      A       430.008408           2.099953                 0.114137   
              B      3742.579863           2.099953                 0.118065   
EdgeGRU       A      3381.032179           2.318100                 0.181831   
              B      2443.806580           2.318100                 0.190306   
StaticGNN     A      5005.872062           2.313602                 0.260097   
              B      4462.034002           2.313602                 0.273281   
ST-GNN        A      2914.137743           2.327396                 0.262219   
              B      6340.845746           2.130867                 0.275872   

                    peak_cuda_reserved_gib  oof_packets  \
model_display fold                                        
Edge MLP      A                   0.208984     11533878   
              B                   0.218750      2675496   
EdgeGRU       A                   0.439453     11533878   
              B                   0.355469      2675496   
StaticGNN     A                   0.552734     11533878   
              B                   0.517578      2675496   
ST-GNN        A                   0.558594     11533878   
              B                   0.527344      2675496   

                    pure_inference_seconds  seconds_per_million_packets  
model_display fold                                                       
Edge MLP      A                  49.082787                     4.255532  
              B                  25.904613                     9.682172  
EdgeGRU       A                  95.480353                     8.278252  
              B                  58.126117                    21.725361  
StaticGNN     A                 158.275593                    13.722669  
              B                 100.323997                    37.497345  
ST-GNN        A                 177.934478                    15.427116  
              B                 127.324605                    47.589159

## 9. Review the saved English report

The generated `review.md` is descriptive and explicitly preserves the pending-ablation boundary. It does not automatically approve topology as the central proposal component.

In [10]:
display(Markdown((OUTPUT_DIR / "review.md").read_text(encoding="utf-8")))
print("Comparison status:", comparison_report["status"])
print("Models compared:", comparison_report["models_compared"])
print("Models pending:", comparison_report["models_pending"])
print("Topology claim authorized:", comparison_report["topology_claim_authorized"])
print("Model training performed:", comparison_report["model_training_performed"])
print("Held-out scenarios accessed:", comparison_report["held_out_scenarios_accessed"])
print("Saved report:", OUTPUT_DIR / "comparison_report.json")
print("Saved review:", OUTPUT_DIR / "review.md")

# cAPTure four-model development comparison

Status: **exploratory partial comparison complete**.

This report uses only checksum-verified development OOF predictions from seed 42. It does not access held-out scenarios and does not support a confirmatory claim.

## Ranking metrics

| Model | Hierarchical AP | Hierarchical ROC-AUC |
|---|---|---|
| Edge MLP | 0.959441 | 0.958992 |
| EdgeGRU | 0.993317 | 0.996617 |
| StaticGNN | 0.994474 | 0.996179 |
| ST-GNN | 0.992172 | 0.997241 |

## Primary operational budget

Each model receives its own development-OOF threshold under the frozen budget of one false-alert window per hour. Calibration and reporting use the same OOF predictions, so these values are screening estimates.

| Model | Worst-fold FA/h | Packet F1 | Packet F2 | Iteration coverage | Timely coverage | Preterminal scenario coverage |
|---|---|---|---|---|---|---|
| Edge MLP | 0.0000 | 0.3533 | 0.3004 | 0.1703 | 0.1353 | 0.5000 |
| EdgeGRU | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| StaticGNN | 0.9464 | 0.8085 | 0.7408 | 0.7405 | 0.7142 | 1.0000 |
| ST-GNN | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |

## False-alert-budget sensitivity

The one-per-hour budget remains primary. The one-per-12-hours and one-per-five-minutes budgets were frozen before this comparison and are reported only as sensitivity analyses; they do not replace the primary budget.

| Model | Budget | Worst-fold FA/h | Packet recall | Packet F1 | Packet F2 | Timely coverage |
|---|---|---|---|---|---|---|
| Edge MLP | one_per_hour | 0.0000 | 0.2732 | 0.3533 | 0.3004 | 0.1353 |
| Edge MLP | one_per_12_hours | 0.0000 | 0.2732 | 0.3533 | 0.3004 | 0.1353 |
| Edge MLP | one_per_5_minutes | 8.5932 | 0.7110 | 0.8169 | 0.7486 | 0.8976 |
| EdgeGRU | one_per_hour | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| EdgeGRU | one_per_12_hours | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| EdgeGRU | one_per_5_minutes | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| StaticGNN | one_per_hour | 0.9464 | 0.7052 | 0.8085 | 0.7408 | 0.7142 |
| StaticGNN | one_per_12_hours | 0.0000 | 0.1867 | 0.2657 | 0.2116 | 0.1750 |
| StaticGNN | one_per_5_minutes | 11.9708 | 0.7930 | 0.8650 | 0.8182 | 0.7970 |
| ST-GNN | one_per_hour | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| ST-GNN | one_per_12_hours | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| ST-GNN | one_per_5_minutes | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |

## Predeclared contrasts

| Comparison | Metric | Delta | Positive scenarios | Negative scenarios |
|---|---|---|---|---|
| temporal_memory_without_gat | average_precision | 0.033876 | 5 | 0 |
| temporal_memory_without_gat | timely_iteration_rate | -0.135268 | 0 | 2 |
| current_message_passing_without_memory | average_precision | 0.035033 | 5 | 0 |
| current_message_passing_without_memory | timely_iteration_rate | 0.578890 | 5 | 0 |
| temporal_memory_with_gat | average_precision | -0.002302 | 3 | 2 |
| temporal_memory_with_gat | timely_iteration_rate | -0.714158 | 0 | 5 |
| gat_signal_in_temporal_models | average_precision | -0.001145 | 2 | 3 |
| gat_signal_in_temporal_models | timely_iteration_rate | 0.000000 | 0 | 0 |

## Resource summary

| Model | Total job wall seconds | Inference s/M packets | Peak host GiB | Peak CUDA GiB |
|---|---|---|---|---|
| Edge MLP | 4172.588 | 5.277 | 2.100 | 0.118 |
| EdgeGRU | 5824.839 | 10.810 | 2.318 | 0.190 |
| StaticGNN | 9467.906 | 18.199 | 2.314 | 0.273 |
| ST-GNN | 9254.983 | 21.483 | 2.327 | 0.276 |

## Interpretation boundary

- `StaticGNN - Edge MLP` is the current aggregation/message-passing signal without recurrent memory.
- `ST-GNN - EdgeGRU` is the temporal comparison, but it retains residual architecture differences.
- `ST-GNN without GAT message passing` is still pending. It retains endpoint aggregation and per-node memory; it is not a model without all relational structure.
- The no-direct-edge-attribute ablation and shifted-window-origin sensitivity are also pending.
- A topology-centered proposal cannot be approved from this partial comparison alone.

Test1, Test2, `train_pub_exf`, and `train_user_prop` were not accessed.


Comparison status: exploratory_four_model_development_comparison_complete
Models compared: ['edge_mlp', 'edge_gru', 'static_gnn', 'st_gnn']
Models pending: ['st_gnn_without_gat', 'st_gnn_without_direct_edge_attr']
Topology claim authorized: False
Model training performed: False
Held-out scenarios accessed: False
Saved report: /content/drive/MyDrive/capture_gate0/graph_comparison_runs/20261001T153351_608797Z_graph_four_model_comparison/comparison_report.json
Saved review: /content/drive/MyDrive/capture_gate0/graph_comparison_runs/20261001T153351_608797Z_graph_four_model_comparison/review.md
